In [4]:
# ==========================================
# 12주차: 민감도분석 노트북 (code/12_sensitivity.ipynb)
# ==========================================
!pip install pyreadstat statsmodels

from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

drive.mount('/content/drive')

# 1. 데이터 로드 (11주차와 동일)
DATA_DIR = '/content/drive/MyDrive/hdac_data'
target_cols = [
    'age', 'sex', 'kstrata', 'psu', 'wt_ntr',
    'genertn', 'HEI', 'incm', 'edu', 'BD1_11', 'pa_aerobic'
]

df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols)
if 'HEI' in df.columns:
    df['HE_khei'] = df['HEI']

# 2. 10~11주차 전처리 동일 적용
df['genertn_rc'] = df['genertn'].apply(lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan))
df['BD1_11_rc'] = df['BD1_11'].replace(9.0, np.nan)

cond_adult = df['age'] >= 20
cond_wt = df['wt_ntr'].notnull()
cond_exp = df['genertn_rc'].notnull()
cond_out = df['HE_khei'].notnull()
df['subpop_t1'] = np.where(cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0)

covars = ['age', 'sex', 'incm', 'edu']
cond_covars = df[covars].notnull().all(axis=1)
df['subpop_reg'] = np.where((df['subpop_t1'] == 1.0) & cond_covars, 1.0, 0.0)

# 3. 설계 기반 회귀 함수 (층 kstrata + 집락 psu + 가중치 wt_ntr)
def fit_design_wls_domain(formula, data, domain_col='subpop_reg',
                          weight_col='wt_ntr', strata_col='kstrata', psu_col='psu'):
    dom_mask = (data[domain_col] == 1.0).to_numpy()
    sub = data.loc[dom_mask].copy()

    mod = smf.wls(formula, data=sub, weights=sub[weight_col])
    res = mod.fit()

    X_sub = np.asarray(mod.exog, dtype=float)
    w_sub = np.asarray(mod.weights, dtype=float)
    u_sub = np.asarray(res.resid * w_sub, dtype=float)[:, None] * X_sub

    n_total = len(data)
    p = X_sub.shape[1]
    u_full = np.zeros((n_total, p), dtype=float)
    u_full[dom_mask] = u_sub

    xtwx = (X_sub.T * w_sub) @ X_sub
    bread = np.linalg.pinv(xtwx)

    strata = data[strata_col].to_numpy()
    psu = data[psu_col].to_numpy()

    meat = np.zeros((p, p), dtype=float)
    total_psu = 0
    for h in np.unique(strata):
        idx_h = np.where(strata == h)[0]
        psu_h = psu[idx_h]
        u_h = u_full[idx_h]
        unique_psu_h = np.unique(psu_h)
        m_h = len(unique_psu_h)
        total_psu += m_h
        if m_h > 1:
            psu_sums = np.vstack([u_h[psu_h == c].sum(axis=0) for c in unique_psu_h])
            diff = psu_sums - psu_sums.mean(axis=0, keepdims=True)
            meat += (m_h / (m_h - 1.0)) * (diff.T @ diff)

    cov_design = bread @ meat @ bread
    se = np.sqrt(np.diag(cov_design))
    df_design = total_psu - len(np.unique(strata))

    beta = res.params
    t_stats = beta.to_numpy() / se
    pvals = 2 * stats.t.sf(np.abs(t_stats), df=df_design)
    t_crit = stats.t.ppf(0.975, df=df_design)
    ci_low = beta.to_numpy() - t_crit * se
    ci_high = beta.to_numpy() + t_crit * se

    return pd.DataFrame({
        'beta': beta, 'se': se, 'ci_low': ci_low,
        'ci_high': ci_high, 'p_value': pvals, 'df': df_design
    }, index=beta.index), int(dom_mask.sum())

formula_m2 = "HE_khei ~ genertn_rc + age + C(sex) + C(incm) + C(edu)"
tab_main, n_main = fit_design_wls_domain(formula_m2, df, domain_col='subpop_reg')
r_main = tab_main.loc['genertn_rc']

print(f"[주 분석 재현 확인] n={n_main:,}, beta={r_main['beta']:.2f}, SE={r_main['se']:.2f}, "
      f"95% CI=[{r_main['ci_low']:.2f}, {r_main['ci_high']:.2f}], p={r_main['p_value']:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[주 분석 재현 확인] n=5,652, beta=-2.49, SE=0.49, 95% CI=[-3.45, -1.52], p=0.0000


In [5]:
# ------------------------------------------
# S2: SE 방법 비교 (집락만 반영한 군집 강건 SE 근사 vs 주 분석 설계 기반 SE)
# ------------------------------------------
sub_main = df.loc[df['subpop_reg'] == 1.0].copy()
fit_s2 = smf.wls(formula_m2, data=sub_main, weights=sub_main['wt_ntr']).fit(
    cov_type='cluster', cov_kwds={'groups': sub_main['psu']}
)
b_s2 = fit_s2.params['genertn_rc']
se_s2 = fit_s2.bse['genertn_rc']
ci_s2_low, ci_s2_high = fit_s2.conf_int().loc['genertn_rc']
p_s2 = fit_s2.pvalues['genertn_rc']

# ------------------------------------------
# S3: 연령 도메인 (65세 이상 vs 65세 미만)
# ------------------------------------------
df['subpop_age65_ge'] = np.where((df['subpop_reg'] == 1.0) & (df['age'] >= 65), 1.0, 0.0)
df['subpop_age65_lt'] = np.where((df['subpop_reg'] == 1.0) & (df['age'] < 65), 1.0, 0.0)

tab_s3a, n_s3a = fit_design_wls_domain(formula_m2, df, domain_col='subpop_age65_ge')
tab_s3b, n_s3b = fit_design_wls_domain(formula_m2, df, domain_col='subpop_age65_lt')

r_s3a = tab_s3a.loc['genertn_rc']
r_s3b = tab_s3b.loc['genertn_rc']

def fmt_p(p):
    return "< 0.0001" if p < 0.0001 else f"{p:.4f}"

def fmt_row(label, n_val, b_val, se_val, lo_val, hi_val, p_val, b_ref):
    diff = b_val - b_ref
    return {
        "분석 구분": label,
        "분석 인원 (n)": f"{n_val:,}",
        "계수 (β)": f"{b_val:.2f}",
        "SE": f"{se_val:.2f}",
        "95% 신뢰구간": f"[{lo_val:.2f}, {hi_val:.2f}]",
        "p값": fmt_p(p_val),
        "주 분석과의 차이": "기준 (0.00)" if label.startswith("주 분석") else f"Δβ = {diff:+.2f}"
    }

table3_df = pd.DataFrame([
    fmt_row("주 분석 (모형 2, 설계 기반 SE)", n_main, r_main['beta'], r_main['se'],
            r_main['ci_low'], r_main['ci_high'], r_main['p_value'], r_main['beta']),
    fmt_row("S2. 군집 강건 SE (psu만 반영 근사)", len(sub_main), b_s2, se_s2,
            ci_s2_low, ci_s2_high, p_s2, r_main['beta']),
    fmt_row("S3a. 연령 도메인: 65세 이상", n_s3a, r_s3a['beta'], r_s3a['se'],
            r_s3a['ci_low'], r_s3a['ci_high'], r_s3a['p_value'], r_main['beta']),
    fmt_row("S3b. 연령 도메인: 65세 미만", n_s3b, r_s3b['beta'], r_s3b['se'],
            r_s3b['ci_low'], r_s3b['ci_high'], r_s3b['p_value'], r_main['beta']),
])

display(table3_df)

,분석 구분,분석 인원 (n),계수 (β),SE,95% 신뢰구간,p값,주 분석과의 차이
0,"주 분석 (모형 2, 설계 기반 SE)","5,652",-2.49,0.49,"[-3.45, -1.52]",< 0.0001,기준 (0.00)
1,S2. 군집 강건 SE (psu만 반영 근사),"5,652",-2.49,0.47,"[-3.41, -1.56]",< 0.0001,Δβ = +0.00
2,S3a. 연령 도메인: 65세 이상,"1,791",-1.50,0.72,"[-2.92, -0.07]",0.0399,Δβ = +0.99
3,S3b. 연령 도메인: 65세 미만,"3,861",-3.03,0.61,"[-4.23, -1.82]",< 0.0001,Δβ = -0.54
